**Set the default lakehouse**

```
        %%configure
        {
            "defaultLakehouse": {  # This overwrites the default lakehouse for current session        "name": "your-lakehouse-name",
                #"id": "<(optional) lakehouse-id>",
                #"workspaceId": "<(optional) workspace-id-that-contains-the-lakehouse>" # Add workspace ID if it's from another workspace
            }
        }
```

In [1]:
%%configure
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, -1, Finished, Available, Finished)

In [2]:
%run DE_NB_000_UTILS_Process_Excel

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 15, Finished, Available, Finished)

In [3]:
import yaml
import sempy.fabric as fabric
from pyspark.sql.functions import col
from pyspark.sql import DataFrame

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 16, Finished, Available, Finished)

In [4]:
def remove_blanks_from_column_names(df: DataFrame) -> DataFrame:
    """
    Removes blanks (spaces) from column names in a PySpark DataFrame.

    :param df: Input PySpark DataFrame
    :return: DataFrame with blanks removed from column names
    """
    new_column_names = [col.replace(' ', '') for col in df.columns]
    for old_name, new_name in zip(df.columns, new_column_names):
        df = df.withColumnRenamed(old_name, new_name)
    return df

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 17, Finished, Available, Finished)

### Get Parameters

In [5]:
#worksheet = 'VAT_Rate_Identifier'
#tablename = 'vatgrouplookup'
#file_name = 'BWINT-001 END source to target mapping - Products.xlsx'
#select_columns = "A,B,C,D"

#worksheet = 'Trade_Terms'
#tablename = 'trandetermslookup'
#file_name = 'BWINT-002 END source to target mapping - Suppliers.xlsx'
#select_columns = "A,B,C,D" 

worksheet = 'Transport_ID'
tablename = 'transportidlookup'
file_name = 'BWINT-004 END source to target mapping - Shipments.xlsx'
select_columns = "A,B,C,D,E" 

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 18, Finished, Available, Finished)

#### Get variables for workspace and lakehouse

In [6]:
workspace_id = fabric.get_workspace_id()
lakehouse_id = fabric.get_lakehouse_id()

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 19, Finished, Available, Finished)

### Set Folders and table-path

In [7]:
lakehouse_files_path = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files'
excel_mapping_folder_path = f'{lakehouse_files_path}/excel_mapping/'

file_path = excel_mapping_folder_path + file_name
tablename = 'DE_LH_100_BondedWarehouse' + '.' + tablename

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 20, Finished, Available, Finished)

### Read the Excel file

In [8]:
df = read_excel_with_trim(file_path, worksheet, select_columns)

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 21, Finished, Available, Finished)

In [9]:
display(df)

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 22, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, d156b844-1a7c-4639-a53f-322515bf719e)

## Clean spaces from column names

In [10]:
df_py = spark.createDataFrame(df)
df_cleaned = remove_blanks_from_column_names(df_py)

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 23, Finished, Available, Finished)

In [11]:
df_cleaned = df_cleaned.withColumnRenamed("Border Transport ID", "Bordertransportid") \
                        .withColumnRenamed("Border Transport Nationality", "Bordertransportnationality") \
                        .withColumnRenamed("Inland Transport ID", "Inlandtransportid")
display(df_cleaned)

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 24, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6a2164b1-cb27-451e-8058-94979ff9f085)

### Write the Lookup to the table

In [12]:
df_cleaned.write.mode("overwrite").option("mergeSchema",False).format("delta").saveAsTable(tablename)

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 25, Finished, Available, Finished)

### Read the table

In [13]:
sql_code = f"SELECT * FROM {tablename} LIMIT 1000"

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 26, Finished, Available, Finished)

In [15]:
df = spark.sql(sql_code)
display(df.count())
display(df)
display(df_cleaned)

StatementMeta(, 067de1cb-bf30-478f-b2e9-f5bfaa9a8f6f, 28, Finished, Available, Finished)

13

SynapseWidget(Synapse.DataFrame, 6a87a266-e4ce-4834-9859-0c66ae1d46b6)

SynapseWidget(Synapse.DataFrame, 262bb520-7ff7-4928-9882-47f26b988614)